# Notebook 05 — End-to-End Clinical Intelligence Demo

Interactive demonstration of the complete CEIS pipeline:
1. **Single Claim Verification**: Multi-question typed outputs
2. **Evidence Span Extraction**: Highlighting supporting vs contradicting text
3. **Long Passage Handling**: Chunking and aggregating extended clinical notes
4. **Contradiction Scanner**: Pairwise matrix over multiple clinical assertions
5. **Evidence Sufficiency Analysis**: Testing multiple claims against one note

In [ ]:
import sys, json
from pathlib import Path
from IPython.display import HTML, display

sys.path.insert(0, str(Path('..').resolve()))
from inference.pipeline import ClinicalEvidencePipeline
from inference.contradiction import ContradictionDetector

## 1 · Initialize Pipeline

In [ ]:
pipe = ClinicalEvidencePipeline(model_path='pavanmantha/mednli_laya')
print('CEIS Pipeline initialized successfully.')

## 2 · Single Verification with Evidence Highlighting

In [ ]:
claim = 'Patient has elevated creatinine indicating renal compromise.'
evidence = 'Admission laboratory work was notable for serum creatinine of 2.1 mg/dL, up from baseline of 0.8 mg/dL.'

res = pipe.verify(claim, evidence)
print(f'Verdict              : {res.verdict.upper()} ({res.confidence:.1%})')
print(f'Explicitly Mentioned : {res.explicitly_mentioned}')
print(f'Needs More Evidence  : {res.needs_more_evidence}')
print(f'Sufficiency Index    : {res.evidence_sufficiency:.2f}')
print(f'Human Review Flag    : {res.human_review_recommended}')
print(f'\nReasoning:\n{res.reasoning}')

print('\nHTML Highlight Preview:')
display(HTML(res.highlighted_evidence_html))

## 3 · Contradiction Detection across Medical Statements

In [ ]:
statements = [
    'Patient is allergic to penicillin with hives upon exposure.',
    'Received IV ampicillin without adverse allergic response.',
    'Blood glucose measured normal at 95 mg/dL fasting.',
    'Diagnosed with acute diabetic ketoacidosis with glucose 480 mg/dL.'
]

detector = ContradictionDetector(pipe)
matrix = detector.detect(statements)

print(f'Statements evaluated: {matrix.n}')
print(f'Flagged Contradictions: {len(matrix.flagged_pairs)}')
print('\n' + detector.summarize(matrix))

## 4 · Evidence Sufficiency Analysis

In [ ]:
clinical_note = ('Patient was admitted with acute dyspnea and productive cough. '
                 'Chest radiography revealed right lower lobe consolidation consistent with pneumonia. '
                 'Started on IV ceftriaxone and azithromycin. Blood pressure 128/82, heart rate 88.')

claims = [
    'Patient has pneumonia.',
    'Patient is prescribed antibiotic therapy.',
    'Patient underwent coronary artery bypass grafting.',
    'Patient has severe hypertension.'
]

suff_results = pipe.analyze_sufficiency(claims, clinical_note)
for s in suff_results:
    print(f'[{s.verdict.upper():12s} | Sufficiency {s.evidence_sufficiency:.2f}]  Claim: {s.claim}')